# Initial Dimension Load – Car Workshop

Run **once** after `create_tables.sql`. Builds the 7 dimension tables and writes
each as a **parquet snapshot** to the external volume on ADLS:
`/Volumes/car_workshop/dim/dim_landing/<table_name>/` (overwrite – safe to re-run).

The generation logic lives in **`simulator/dims_generation.py`** (seed 42, shared
with the local Docker lab); this notebook only provides the Spark writer.

Prerequisite: the `car_workshop.dim.dim_landing` external volume must exist
(`infra/create_external_adls.sql`, section 3). File -> `car_workshop.dim.*`
ingestion is a separate step (see `journal/todo/04-sheets-redesign.md`).

| table | rows |
|---|---|
| dim_locations | 99 (one per city in CITIES) |
| dim_employees | ~1,160 |
| dim_customers | 50,000 |
| dim_vehicles | 65,000 |
| dim_products | ~930 (product names x 2-6 manufacturers) |
| dim_services | 96 (= SERVICE_CATALOGUE) |
| dim_suppliers | 500 |

In [ ]:
%pip install faker

In [ ]:
from simulator.dims_generation import generate_dims
from table_schemas import TABLE_SCHEMAS, schema_to_ddl

CATALOG = 'car_workshop'
DIM_LANDING = f'/Volumes/{CATALOG}/dim/dim_landing'  # external volume on ADLS


def save_dim(pdf, table_name):
    df = spark.createDataFrame(pdf, schema=schema_to_ddl(TABLE_SCHEMAS[table_name]))
    path = f'{DIM_LANDING}/{table_name}'
    df.write.mode('overwrite').parquet(path)
    print(f'  {table_name}: {len(pdf):,} rows -> {path}')


stats = generate_dims(save_dim)  # scale=1.0 -> the full deterministic dataset

## Validation

In [ ]:
for table in ['dim_locations', 'dim_employees', 'dim_customers', 'dim_vehicles',
              'dim_products', 'dim_services', 'dim_suppliers']:
    n = spark.read.parquet(f'{DIM_LANDING}/{table}').count()
    print(f'{table}: {n:,} rows')